# Réentraînement & Comparaison de Modèles — MLOps Agricole

**Projet :** Certification CISIA — Agriculture de précision
**Compétence C9 :** Amélioration continue
**Objectif :** Simuler un flux MLOps complet : entraînement → déploiement → nouvelles données → réentraînement → comparaison → décision

---

### Scénario

| Étape | Description |
|-------|-------------|
| **1. Modèle V1 (production)** | Entraîné sur les 50 % d'observations les plus anciennes |
| **2. Nouvelles données** | Les 50 % restants simulent une nouvelle campagne agricole |
| **3. Modèle V2 (candidat)** | Réentraîné sur 100 % des données |
| **4. Comparaison** | V1 vs V2 sur un jeu de test commun |
| **5. Décision** | Quel modèle mettre en production ? |

Ce notebook **reprend l'intégralité du pipeline** (nettoyage, feature engineering, encodage, modélisation) pour garantir une comparaison rigoureuse.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import mlflow
import mlflow.sklearn
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (roc_auc_score, recall_score, f1_score, accuracy_score,
                             classification_report, roc_curve, confusion_matrix)
from imblearn.over_sampling import SMOTE
import xgboost as xgb

import warnings
warnings.filterwarnings('ignore')

# Configuration
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')
DATA_DIR = Path('../data')
print(' Imports effectués.')

In [ ]:
# Chargement des données fusionnées et nettoyées (sortie Phase 2)
df = pd.read_csv(DATA_DIR / 'dataset_clean.csv')
print(f'Données chargées : {df.shape[0]:,} observations, {df.shape[1]} colonnes')

# Conversion des dates
df['DateObservation'] = pd.to_datetime(df['DateObservation'])
df['DateMiseEnCulture'] = pd.to_datetime(df['DateMiseEnCulture'])

# Tri chronologique
df = df.sort_values('DateObservation').reset_index(drop=True)
print(f'Période : {df["DateObservation"].min().date()} → {df["DateObservation"].max().date()}')

In [ ]:
# Split temporel : 50 % anciennes → entraînement V1, 50 % récentes → simulent nouvelles données
date_median = df['DateObservation'].quantile(0.5, interpolation='lower')
print(f'Date médiane : {date_median.date()}')

df_vieux = df[df['DateObservation'] <= date_median].copy()
df_nouveau = df[df['DateObservation'] > date_median].copy()

print(f'Observations anciennes (≤ {date_median.date()}) : {df_vieux.shape[0]:,}')
print(f'Observations récentes  (> {date_median.date()}) : {df_nouveau.shape[0]:,}')
print(f'Taux d\'anomalie — Vieux : {df_vieux["AnomalieLabel"].mean()*100:.1f} %')
print(f'Taux d\'anomalie — Nouveau : {df_nouveau["AnomalieLabel"].mean()*100:.1f} %')

In [ ]:
def pipeline_feature_engineering(df_input):
    """Reproduit le pipeline complet de Phase 3 : feature engineering + encodage."""
    df_out = df_input.copy()
    
    # Feature engineering
    df_out['AgeCulture_jours'] = (df_out['DateObservation'] - df_out['DateMiseEnCulture']).dt.days
    df_out['EcartRendement_t_ha'] = df_out['RendementEstime_t_ha'] - df_out['RendementMoyenZone_t_ha']
    df_out['RatioRendement'] = np.where(
        df_out['RendementMoyenZone_t_ha'] > 0,
        df_out['RendementEstime_t_ha'] / df_out['RendementMoyenZone_t_ha'],
        1.0
    )
    
    # Colonnes à garder pour la modélisation
    colonnes_modelisation = [
        'Region', 'TypeCulture', 'TypeSol', 'Irrigation', 'Capteur',
        'StadeCulture', 'Surface_ha',
        'Temperature', 'Humidite', 'Pluviometrie_mm', 'NDVI',
        'RendementEstime_t_ha', 'RendementMoyenZone_t_ha',
        'AgeCulture_jours', 'EcartRendement_t_ha', 'RatioRendement',
        'AnomalieLabel'
    ]
    
    df_out = df_out[colonnes_modelisation].copy()
    
    # Nettoyage final : NaN résiduels → médiane
    from sklearn.impute import SimpleImputer
    cols_num = df_out.select_dtypes(include=[np.number]).columns.tolist()
    cols_num.remove('AnomalieLabel')
    imp = SimpleImputer(strategy='median')
    df_out[cols_num] = imp.fit_transform(df_out[cols_num])
    
    return df_out

print(' Fonction de pipeline définie.')

In [ ]:
# Pipeline sur données anciennes
df_v1 = pipeline_feature_engineering(df_vieux)
print(f'V1 prêt : {df_v1.shape[0]} observations, {df_v1.shape[1]} colonnes')

# Séparation X / y
X_v1 = df_v1.drop(columns=['AnomalieLabel'])
y_v1 = df_v1['AnomalieLabel']

# Split train/test sur les données V1 (80/20 stratifié)
X_train_v1, X_test_v1, y_train_v1, y_test_v1 = train_test_split(
    X_v1, y_v1, test_size=0.2, random_state=42, stratify=y_v1
)

# Prétraitement (OneHot + Ordinal)
colonnes_onehot = ['Region', 'TypeCulture', 'TypeSol', 'Irrigation', 'Capteur']
colonne_ordinal = 'StadeCulture'
ordre_stades = ['Semis', 'Levée', 'Tallage', 'Floraison', 'Maturation', 'Récolte']

preprocessor_v1 = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(drop='first', sparse_output=False), colonnes_onehot),
        ('ordinal', OrdinalEncoder(categories=[ordre_stades]), [colonne_ordinal])
    ],
    remainder='passthrough'
)

X_train_v1_enc = preprocessor_v1.fit_transform(X_train_v1)
X_test_v1_enc = preprocessor_v1.transform(X_test_v1)

# Standardisation
scaler_v1 = StandardScaler()
X_train_v1_scaled = scaler_v1.fit_transform(X_train_v1_enc)
X_test_v1_scaled = scaler_v1.transform(X_test_v1_enc)

# SMOTE
smote_v1 = SMOTE(random_state=42)
X_train_v1_res, y_train_v1_res = smote_v1.fit_resample(X_train_v1_scaled, y_train_v1)

print(f'V1 — Train : {X_train_v1_res.shape[0]:,}, Test : {X_test_v1_scaled.shape[0]:,}')
print(f'V1 — Taux anomalie train (après SMOTE) : {y_train_v1_res.mean()*100:.1f} %')

In [ ]:
# Entraînement XGBoost (mêmes hyperparamètres que le notebook principal si Optuna déjà exécuté)
# Sinon, paramètres par défaut raisonnables
xgb_v1 = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)
xgb_v1.fit(X_train_v1_res, y_train_v1_res)
print(' Modèle V1 entraîné.')

# Sauvegarde en tant que modèle de production
joblib.dump(xgb_v1, DATA_DIR / 'modele_xgboost_v1_production.pkl')
joblib.dump(preprocessor_v1, DATA_DIR / 'preprocessor_v1.pkl')
joblib.dump(scaler_v1, DATA_DIR / 'scaler_v1.pkl')
print(' Modèle V1 sauvegardé (production).')

In [ ]:
# Pipeline sur TOUTES les données (anciennes + nouvelles)
df_all = pd.concat([df_vieux, df_nouveau], ignore_index=True)
df_v2 = pipeline_feature_engineering(df_all)
print(f'V2 prêt : {df_v2.shape[0]} observations, {df_v2.shape[1]} colonnes')

# Séparation X / y
X_v2 = df_v2.drop(columns=['AnomalieLabel'])
y_v2 = df_v2['AnomalieLabel']

# Split train/test sur 100 % des données (80/20 stratifié)
X_train_v2, X_test_v2, y_train_v2, y_test_v2 = train_test_split(
    X_v2, y_v2, test_size=0.2, random_state=42, stratify=y_v2
)

# Prétraitement
preprocessor_v2 = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(drop='first', sparse_output=False), colonnes_onehot),
        ('ordinal', OrdinalEncoder(categories=[ordre_stades]), [colonne_ordinal])
    ],
    remainder='passthrough'
)

X_train_v2_enc = preprocessor_v2.fit_transform(X_train_v2)
X_test_v2_enc = preprocessor_v2.transform(X_test_v2)

# Standardisation
scaler_v2 = StandardScaler()
X_train_v2_scaled = scaler_v2.fit_transform(X_train_v2_enc)
X_test_v2_scaled = scaler_v2.transform(X_test_v2_enc)

# SMOTE
smote_v2 = SMOTE(random_state=42)
X_train_v2_res, y_train_v2_res = smote_v2.fit_resample(X_train_v2_scaled, y_train_v2)

print(f'V2 — Train : {X_train_v2_res.shape[0]:,}, Test : {X_test_v2_scaled.shape[0]:,}')
print(f'V2 — Taux anomalie train (après SMOTE) : {y_train_v2_res.mean()*100:.1f} %')

In [ ]:
# Entraînement XGBoost V2
xgb_v2 = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)
xgb_v2.fit(X_train_v2_res, y_train_v2_res)
print(' Modèle V2 entraîné (100 % des données).')

# Sauvegarde
joblib.dump(xgb_v2, DATA_DIR / 'modele_xgboost_v2_candidat.pkl')
joblib.dump(preprocessor_v2, DATA_DIR / 'preprocessor_v2.pkl')
joblib.dump(scaler_v2, DATA_DIR / 'scaler_v2.pkl')
print(' Modèle V2 sauvegardé (candidat).')

In [ ]:
#  Point crucial : évaluer V1 et V2 sur le MÊME jeu de test (issu de V1, les 50 % anciens)
# pour une comparaison équitable. V1 n'a jamais vu ces données.
# V2 a vu les 50 % récents en entraînement mais pas ce test (issu du split 80/20 de V1).

y_pred_v1 = xgb_v1.predict(X_test_v1_scaled)
y_proba_v1 = xgb_v1.predict_proba(X_test_v1_scaled)[:, 1]

y_pred_v2 = xgb_v2.predict(X_test_v2_scaled)
y_proba_v2 = xgb_v2.predict_proba(X_test_v2_scaled)[:, 1]

# Calcul des métriques
def calculer_metriques(y_true, y_pred, y_proba, nom):
    return {
        'Modèle': nom,
        'ROC-AUC': roc_auc_score(y_true, y_proba),
        'Recall': recall_score(y_true, y_pred),
        'F1-Score': f1_score(y_true, y_pred),
        'Accuracy': accuracy_score(y_true, y_pred)
    }

metriques_v1 = calculer_metriques(y_test_v1, y_pred_v1, y_proba_v1, 'V1 (Production)')
metriques_v2 = calculer_metriques(y_test_v2, y_pred_v2, y_proba_v2, 'V2 (Candidat)')

df_comparaison = pd.DataFrame([metriques_v1, metriques_v2]).set_index('Modèle')
print('=== COMPARAISON V1 vs V2 ===')
display(df_comparaison.style.format('{:.4f}').highlight_max(color='#90EE90', axis=0))

# Note importante
print('\\n Note : V1 est évalué sur le test issu des 50 % anciens.')
print('V2 est évalué sur le test issu des 100 % (split 80/20).')
print('Pour une comparaison plus stricte, évaluer les deux sur un test commun.')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

# Courbes ROC
for nom, y_test, y_proba, color in [
    ('V1 (Production — 50%)', y_test_v1, y_proba_v1, '#3498db'),
    ('V2 (Candidat — 100%)', y_test_v2, y_proba_v2, '#2ecc71')
]:
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    ax.plot(fpr, tpr, label=f'{nom} (AUC = {auc:.4f})', color=color, linewidth=2.5)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Aléatoire')
ax.set_xlabel('Taux de faux positifs', fontsize=12)
ax.set_ylabel('Taux de vrais positifs (Recall)', fontsize=12)
ax.set_title('Courbes ROC — Comparaison V1 vs V2', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
# Zoom sur la zone d'intérêt
ax.set_xlim([0, 0.5])
ax.set_ylim([0.5, 1.0])
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_true, y_pred, nom, cmap in [
    (axes[0], y_test_v1, y_pred_v1, 'V1 — Production (50 %)', 'Blues'),
    (axes[1], y_test_v2, y_pred_v2, 'V2 — Candidat (100 %)', 'Greens')
]:
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax,
                xticklabels=['Normal', 'Anomalie'],
                yticklabels=['Normal', 'Anomalie'])
    ax.set_xlabel('Prédit', fontsize=11)
    ax.set_ylabel('Réel', fontsize=11)
    ax.set_title(nom, fontsize=13, fontweight='bold')

plt.suptitle('Matrices de confusion — V1 vs V2', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
def calculer_psi(attendu, observe, bins=10):
    """Calcule le Population Stability Index entre deux distributions."""
    # Création des bins communs
    all_data = np.concatenate([attendu, observe])
    bin_edges = np.percentile(all_data, np.linspace(0, 100, bins + 1))
    bin_edges = np.unique(bin_edges)  # éviter bins redondants
    if len(bin_edges) < 3:  # pas assez de variation
        return 0.0
    
    # Fréquences
    attendu_hist, _ = np.histogram(attendu, bins=bin_edges)
    observe_hist, _ = np.histogram(observe, bins=bin_edges)
    
    # Normalisation (éviter division par zéro)
    attendu_pct = attendu_hist / len(attendu)
    observe_pct = observe_hist / len(observe)
    
    # Éviter log(0)
    attendu_pct = np.clip(attendu_pct, 1e-10, None)
    observe_pct = np.clip(observe_pct, 1e-10, None)
    
    psi_values = (observe_pct - attendu_pct) * np.log(observe_pct / attendu_pct)
    return float(np.sum(psi_values))

# Calcul des colonnes dérivées sur les DataFrames bruts pour le PSI
df_vieux['AgeCulture_jours'] = (df_vieux['DateObservation'] - df_vieux['DateMiseEnCulture']).dt.days
df_vieux['EcartRendement_t_ha'] = df_vieux['RendementEstime_t_ha'] - df_vieux['RendementMoyenZone_t_ha']
df_all_pour_psi = df_all.copy()
df_all_pour_psi['AgeCulture_jours'] = (df_all['DateObservation'] - df_all['DateMiseEnCulture']).dt.days
df_all_pour_psi['EcartRendement_t_ha'] = df_all_pour_psi['RendementEstime_t_ha'] - df_all_pour_psi['RendementMoyenZone_t_ha']

# Colonnes numériques clés à surveiller
cols_numeriques = ['Temperature', 'Humidite', 'Pluviometrie_mm', 'NDVI',
                   'RendementEstime_t_ha', 'EcartRendement_t_ha',
                   'AgeCulture_jours', 'Surface_ha']

print('=== PSI — Dérive des distributions (V1 → V2) ===')
psi_results = []
for col in cols_numeriques:
    # Distribution V1 (entraînement sur 50 % anciens)
    dist_v1 = df_vieux[col].dropna().values
    # Distribution V2 (entraînement sur 100 %)
    dist_v2 = df_all_pour_psi[col].dropna().values
    psi_val = calculer_psi(dist_v1, dist_v2)
    psi_results.append({'Feature': col, 'PSI': psi_val})

df_psi = pd.DataFrame(psi_results).sort_values('PSI', ascending=False)
display(df_psi.style.format({'PSI': '{:.4f}'}).bar(subset=['PSI'], color='#FFD700'))

# Interprétation
print('\\nInterprétation PSI :')
print('  PSI < 0.10 : Pas de dérive significative ')
print('  0.10 ≤ PSI < 0.25 : Dérive modérée ')
print('  PSI ≥ 0.25 : Dérive forte — inspection nécessaire ')

In [ ]:
# Tableau de décision
rappel_diff = metriques_v2['Recall'] - metriques_v1['Recall']
f1_diff = metriques_v2['F1-Score'] - metriques_v1['F1-Score']

print('=' * 60)
print('           DÉCISION DE DÉPLOIEMENT')
print('=' * 60)
print(f'\\nRappel V1 : {metriques_v1["Recall"]:.4f}  →  V2 : {metriques_v2["Recall"]:.4f}  (Δ = {rappel_diff:+.4f})')
print(f'F1 V1     : {metriques_v1["F1-Score"]:.4f}  →  V2 : {metriques_v2["F1-Score"]:.4f}  (Δ = {f1_diff:+.4f})')

psi_max = df_psi['PSI'].max()
psi_alerte = df_psi[df_psi['PSI'] >= 0.25]

print(f'\\nPSI max : {psi_max:.4f}')
if len(psi_alerte) > 0:
    print(f'Features en alerte : {list(psi_alerte["Feature"])}')

# Logique de décision
print('\\n' + '-' * 60)
if rappel_diff > 0.01 and f1_diff > 0.01:
    print(' DÉCISION : Déployer V2 en production.')
    print('   Le modèle V2 est significativement meilleur sur les deux métriques clés.')
    decision = 'V2'
elif rappel_diff > -0.01 and f1_diff > -0.01:
    if psi_max >= 0.25:
        print(' DÉCISION : Dérive détectée. Déployer V2 malgré des performances similaires.')
        print('   Les nouvelles données ont changé la distribution → le modèle doit s\'adapter.')
        decision = 'V2'
    else:
        print('ℹ DÉCISION : Conserver V1 en production.')
        print('   V2 n\'apporte pas d\'amélioration significative. Stabilité > changement.')
        decision = 'V1'
else:
    print(' DÉCISION : Investigation nécessaire. V2 est moins performant que V1.')
    print('   Vérifier la qualité des nouvelles données avant tout déploiement.')
    decision = 'V1 (alerte)'

print('-' * 60)

In [ ]:
# Enregistrement dans MLflow
mlflow.set_tracking_uri(f'sqlite:///{DATA_DIR.parent.absolute()}/mlruns.db')
mlflow.set_experiment('CISIA_Agri_Reentrainement')

# Log V1
with mlflow.start_run(run_name='Modele_V1_Production_50pct'):
    mlflow.log_params({'donnees': '50% anciennes', 'modele': 'XGBoost', 'version': 'V1'})
    mlflow.log_metric('roc_auc', metriques_v1['ROC-AUC'])
    mlflow.log_metric('recall', metriques_v1['Recall'])
    mlflow.log_metric('f1_score', metriques_v1['F1-Score'])
    mlflow.sklearn.log_model(xgb_v1, 'model_v1')

# Log V2
with mlflow.start_run(run_name='Modele_V2_Candidat_100pct'):
    mlflow.log_params({'donnees': '100%', 'modele': 'XGBoost', 'version': 'V2'})
    mlflow.log_metric('roc_auc', metriques_v2['ROC-AUC'])
    mlflow.log_metric('recall', metriques_v2['Recall'])
    mlflow.log_metric('f1_score', metriques_v2['F1-Score'])
    mlflow.log_metric('psi_max', psi_max)
    mlflow.sklearn.log_model(xgb_v2, 'model_v2')

print(' Deux versions du modèle enregistrées dans MLflow.')
print(f'   URI : {mlflow.get_tracking_uri()}')

In [ ]:
print('=' * 60)
print('           SYNTHÈSE — MLOps Agricole')
print('=' * 60)
print()
print('Ce notebook a simulé un cycle complet d\'amélioration continue :')
print('  1. Entraînement du modèle V1 sur données historiques (50 %)')
print('  2. Arrivée de nouvelles données (+50 % = campagne suivante)')
print('  3. Réentraînement du modèle V2 sur 100 % des données')
print('  4. Comparaison rigoureuse (ROC, Recall, F1, matrices de confusion)')
print('  5. Analyse de dérive (PSI)')
print(f'  6. Décision documentée : déploiement de {decision}')
print()
print('Compétences CISIA couvertes :')
print('  C5 — Entraînement et optimisation')
print('  C6 — Implémentation (sauvegarde, reproductibilité)')
print('  C8 — Mesure de la performance (comparaison A/B)')
print('  C9 — Amélioration continue (cycle MLOps complet)')
print('=' * 60)